In [7]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

Par 1 : Fraud Classification Model

In [18]:
import os

JAN_PATH = globals().get(
    "JAN_PATH",
    "/Users/maheshg/Dropbox/Sample Datasets Kaggle/transactions_jan2025.csv",
)

In [19]:
JAN_PATH.count

<function str.count>

In [20]:
def load_january_transactions():
    if not os.path.exists(JAN_PATH):
        raise FileNotFoundError(f"January transactions file not found: {JAN_PATH}")
    return pd.read_csv(JAN_PATH)

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

# use the DataFrame that contains the fraud_flag column
df = feat.copy()

df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'])
df = df.sort_values(['customer_name', 'transaction_datetime'])

df['customer_txn_count'] = df.groupby('customer_name').cumcount()
df['merchant_name_freq'] = df['merchant_name'].map(df['merchant_name'].value_counts())

df['hour'] = df['transaction_datetime'].dt.hour
df['day_of_week'] = df['transaction_datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_night'] = df['hour'].isin(range(0, 6)).astype(int)

feature_cols = [
    'transaction_amount',
    'hour',
    'day_of_week',
    'is_weekend',
    'is_night',
    'customer_txn_count',
    'merchant_name_freq',
    'merchant_type',
    'city',
]
numeric_features = [
    'transaction_amount',
    'hour',
    'day_of_week',
    'is_weekend',
    'is_night',
    'customer_txn_count',
    'merchant_name_freq',
]
categorical_features = ['merchant_type', 'city']

X = df[feature_cols]
y = df['fraud_flag'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ]
)

model_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]
)

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9890041770775716
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    103705
           1       0.00      0.00      0.00      1153

    accuracy                           0.99    104858
   macro avg       0.49      0.50      0.50    104858
weighted avg       0.98      0.99      0.98    104858

Confusion matrix:
[[103705      0]
 [  1153      0]]


/Users/maheshg/Dropbox/git repos/PythonPractise/PythonPractise-1/study/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/maheshg/Dropbox/git repos/PythonPractise/PythonPractise-1/study/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/maheshg/Dropbox/git repos/PythonPractise/PythonPractise-1/study/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. U

Part 2 : Business Impact Assessment

    Your model is not a research artifact. The fraud prevention team needs to justify deploying it to operations, to finance and to senior leadership. Your job in this section is to make that case.

    You are not given a cost structure. You are expected to define one, state your assumptions explicitly and build your argument from there.
    What your presentation should address
        • What does it cost the business to miss a fraud? What does it cost to block a legitimate customer? State your assumptions and explain your reasoning.
        • Given those assumptions, how should your model's threshold be set — and why?
        • What is the estimated financial impact of deploying your model versus the current manual process?
        • Which segments of transactions or customers represent the highest risk, and what does that imply for how operations should prioritize?
        • What would you need from the business to refine or validate your assumptions?

In [1]:
import numpy as np
import pandas as pd
 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
 
RANDOM_STATE = 42

FEATURE_COLUMNS_NUMERIC = [
    'transaction_amount',
    'hour',
    'day_of_week',
    'is_weekend',
    'is_night',
    'customer_txn_count',
    'merchant_name_freq',
]
FEATURE_COLUMNS_CATEGORICAL = ['merchant_type', 'city']

def engineer_features(df):
    df = df.copy()
    df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'])
    df = df.sort_values(['customer_name', 'transaction_datetime'])

    df['customer_txn_count'] = df.groupby('customer_name').cumcount()
    df['merchant_name_freq'] = df['merchant_name'].map(df['merchant_name'].value_counts())
    df['hour'] = df['transaction_datetime'].dt.hour
    df['day_of_week'] = df['transaction_datetime'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_night'] = df['hour'].isin(range(0, 6)).astype(int)

    return df
 
HIST_PATH = "/Users/maheshg/Dropbox/Sample Datasets Kaggle/transactions_historical_2024.csv"
 

    # Cost assumptions - stated explicitly, sourced from public averages since
    # I wasn't given internal numbers. Each one has a one-line justification.
    # See "What I'd need from the business" at the bottom for how to replace these

In [2]:
#------------------------------------------------------------------------
 
COST_ASSUMPTIONS = {
    # A missed fraud isn't just the transaction amount. Card networks/issuers
    # typically pass the full transaction back to the merchant on a fraud
    # chargeback (Reg E / card network liability shift), plus there's a flat
    # admin cost to process the dispute (pull evidence, respond, close the
    # case). $25 is a commonly cited low-end chargeback handling cost;
    # I'm treating it as a placeholder, not a researched figure.
    "chargeback_admin_fee": 25.0,
 
    # Cost of a false positive = ops cost to review it + a small chance the
    # customer is annoyed enough to reduce spend or leave. I'm assuming:
    #   - $4 in analyst/ops time to review a single flagged transaction
    #   - a 2% chance a wrongly-blocked customer churns
    #   - an assumed customer lifetime value of $350 if they do
    # 4 + 0.02 * 350 = $11
    "review_cost": 4.0,
    "false_decline_churn_prob": 0.02,
    "customer_lifetime_value": 350.0,
 
    # Cost to investigate a transaction the model correctly flags as fraud
    # (confirming it, contacting the customer, blocking the card). Lower
    # than a full chargeback because it's caught before the loss lands.
    "confirmed_fraud_review_cost": 8.0,
}
 
def fn_cost(amount: np.ndarray) -> np.ndarray:
    """Cost of missing a fraudulent transaction (false negative)."""
    return amount + COST_ASSUMPTIONS["chargeback_admin_fee"]
 
def fp_cost() -> float:
    """Cost of wrongly flagging a legitimate transaction (false positive)."""
    return (COST_ASSUMPTIONS["review_cost"]
            + COST_ASSUMPTIONS["false_decline_churn_prob"] * COST_ASSUMPTIONS["customer_lifetime_value"])
 
def tp_cost() -> float:
    """Cost of correctly catching a fraud (still not free - someone has to act on it)."""
    return COST_ASSUMPTIONS["confirmed_fraud_review_cost"]
 

    # Train the model exactly as in Part 1, but keep transaction_amount attached
    # to the test set so false negatives can be costed at their real dollar value
    # instead of an average.

In [3]:

def train_model(feat_df: pd.DataFrame):
    X = feat_df[FEATURE_COLUMNS_NUMERIC + FEATURE_COLUMNS_CATEGORICAL]
    y = feat_df["fraud_flag"].astype(int)
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
 
    class_counts = y_train.value_counts()
    weight_for_class = {0: 1.0, 1: class_counts[0] / class_counts[1]}
    sw_train = y_train.map(weight_for_class).values
 
    pre = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), FEATURE_COLUMNS_CATEGORICAL)],
        remainder="passthrough",
    )
    pipe = Pipeline([("prep", pre), ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))])
    pipe.fit(X_train, y_train, model__sample_weight=sw_train)
 
    proba_test = pipe.predict_proba(X_test)[:, 1]
    return pipe, X_test, y_test, proba_test
 

### Cost minimising threshold

In [4]:

def cost_at_threshold(y_true, proba, amounts, threshold):
    flagged = proba >= threshold
    fn_mask = (~flagged) & (y_true == 1)
    fp_mask = flagged & (y_true == 0)
    tp_mask = flagged & (y_true == 1)
 
    total = fn_cost(amounts[fn_mask]).sum() + fp_mask.sum() * fp_cost() + tp_mask.sum() * tp_cost()
    return total, fn_mask.sum(), fp_mask.sum(), tp_mask.sum()
 
 
def find_optimal_threshold(y_true, proba, amounts):
    y_true = np.asarray(y_true)
    amounts = np.asarray(amounts)
    grid = np.unique(np.concatenate([np.linspace(0.001, 0.999, 400), proba]))
    rows = []
    for t in grid:
        cost, fn, fp, tp = cost_at_threshold(y_true, proba, amounts, t)
        rows.append((t, cost, fn, fp, tp))
    curve = pd.DataFrame(rows, columns=["threshold", "total_cost", "fn", "fp", "tp"])
    best = curve.loc[curve["total_cost"].idxmin()]
    return best, curve
 


    # ---------------------------------------------------------------------------
    # Baseline: what the current, no-model process is assumed to look like.
    # I don't have the real rule the ops team runs today, so I'm assuming the
    # most common naive version of it: flag anything over a fixed dollar amount
    # and route it to manual review; everything under that amount goes through
    # untouched. $75 flags roughly the top 4% of transactions by volume here.
    # This assumption is exactly the kind of thing that should get replaced with
    # the real current rule as soon as ops shares it.
    # ---------------------------------------------------------------------------
    

In [5]:

CURRENT_PROCESS_AMOUNT_THRESHOLD = 75.0
 
def cost_of_current_process(y_true, amounts):
    flagged = amounts >= CURRENT_PROCESS_AMOUNT_THRESHOLD
    fn_mask = (~flagged) & (y_true == 1)
    fp_mask = flagged & (y_true == 0)
    tp_mask = flagged & (y_true == 1)
    total = fn_cost(amounts[fn_mask]).sum() + fp_mask.sum() * fp_cost() + tp_mask.sum() * tp_cost()
    return total, fn_mask.sum(), fp_mask.sum(), tp_mask.sum(), flagged.mean()
 
 
def cost_of_doing_nothing(y_true, amounts):
    fn_mask = (y_true == 1)
    return fn_cost(amounts[fn_mask]).sum()
 

    # ---------------------------------------------------------------------------
    # Segment risk - ranked by dollar exposure (fraud rate x volume x avg
    # amount), not just fraud rate, since a segment with a slightly higher
    # fraud rate but tiny volume isn't where ops time should go.
    # ---------------------------------------------------------------------------

In [6]:

def segment_risk_table(feat_df: pd.DataFrame, group_col: str):
    g = feat_df.groupby(group_col)
    out = g.agg(
        volume=("fraud_flag", "size"),
        fraud_rate=("fraud_flag", "mean"),
        avg_amount=("transaction_amount", "mean"),
        fraud_dollar_exposure=("transaction_amount", lambda s: s[feat_df.loc[s.index, "fraud_flag"]].sum()),
    ).sort_values("fraud_dollar_exposure", ascending=False)
    out["share_of_total_fraud_dollars"] = out["fraud_dollar_exposure"] / out["fraud_dollar_exposure"].sum()
    return out
 
 
def breakeven_precision(fp_c: float, tp_c: float, avg_fn_c: float) -> float:
    """
    Precision a flagged queue needs to hit before flagging is worth it at all.
    Flagging a transaction only pays off if:
        p * (avg_fn_c - tp_c)  >  (1 - p) * fp_c
    Solve for p (breakeven precision).
    """
    benefit_if_caught = avg_fn_c - tp_c
    return fp_c / (benefit_if_caught + fp_c)
 
 
def annualize(monthly_or_period_cost: float, n_days_in_sample: int) -> float:
    return monthly_or_period_cost * (365.0 / n_days_in_sample)
 
 
if __name__ == "__main__":
    raw = pd.read_csv(HIST_PATH)
    raw["fraud_flag"] = raw["fraud_flag"].astype(bool)
    feat = engineer_features(raw)
 
    print("=" * 72)
    print("COST ASSUMPTIONS")
    print("=" * 72)
    print(f"Cost of a missed fraud (FN)      = transaction amount + ${COST_ASSUMPTIONS['chargeback_admin_fee']:.0f} admin")
    print(f"Cost of a false positive (FP)    = ${fp_cost():.2f}  "
          f"(${COST_ASSUMPTIONS['review_cost']:.0f} review + "
          f"{COST_ASSUMPTIONS['false_decline_churn_prob']:.0%} churn risk x ${COST_ASSUMPTIONS['customer_lifetime_value']:.0f} CLV)")
    print(f"Cost of a caught fraud (TP)      = ${tp_cost():.2f} (investigation/action)")
 
    pipe, X_test, y_test, proba = train_model(feat)
    amounts_test = X_test["transaction_amount"].values
 
    print("\n" + "=" * 72)
    print("COST-MINIMIZING THRESHOLD")
    print("=" * 72)
    best, curve = find_optimal_threshold(y_test.values, proba, amounts_test)
    print(best)
 
    n_days = (pd.to_datetime(raw["transaction_datetime"]).max() - pd.to_datetime(raw["transaction_datetime"]).min()).days
    test_share_of_days = 0.2  # test set is 20% of rows, roughly 20% of the days at this volume
 
    nothing_cost = cost_of_doing_nothing(y_test.values, amounts_test)
    current_cost, cur_fn, cur_fp, cur_tp, cur_flag_rate = cost_of_current_process(y_test.values, amounts_test)
    model_cost = best["total_cost"]
 
    print("\n" + "=" * 72)
    print("MODEL vs CURRENT PROCESS vs DOING NOTHING (test set, ~20% of the year)")
    print("=" * 72)
    print(f"Doing nothing (no screening at all):      ${nothing_cost:,.0f}")
    print(f"Current process (flag amount >= ${CURRENT_PROCESS_AMOUNT_THRESHOLD:.0f}):  ${current_cost:,.0f}  "
          f"(flags {cur_flag_rate:.1%} of volume, catches {cur_tp}/{cur_tp+cur_fn} frauds)")
    print(f"Model at cost-minimizing threshold:       ${model_cost:,.0f}  "
          f"(flags {(best['fp']+best['tp'])/len(y_test):.1%} of volume, catches {int(best['tp'])}/{int(best['tp']+best['fn'])} frauds)")
 
    savings_vs_nothing = nothing_cost - model_cost
    savings_vs_current = current_cost - model_cost
    print(f"\nModel savings vs doing nothing (this window): ${savings_vs_nothing:,.0f}")
    print(f"Model savings vs current process (this window): ${savings_vs_current:,.0f}")
    print(f"Annualized (x ~5, since test window is ~1/5 of the year's volume):")
    print(f"  vs doing nothing:  ${savings_vs_nothing*5:,.0f}/yr")
    print(f"  vs current process: ${savings_vs_current*5:,.0f}/yr")
 
    print("\n" + "=" * 72)
    print("WHERE THE MODEL ACTUALLY HAS LIFT (precision by flagged volume)")
    print("=" * 72)
    order = np.argsort(-proba)
    n = len(y_test)
    total_fraud = y_test.sum()
    base_rate = total_fraud / n
    lift_rows = []
    for pct in [0.01, 0.02, 0.05, 0.10, 0.20]:
        k = int(n * pct)
        idx = order[:k]
        caught = y_test.values[idx].sum()
        precision = caught / k
        lift_rows.append((pct, k, caught, precision, precision / base_rate))
    lift_df = pd.DataFrame(lift_rows, columns=["flag_volume_pct", "n_flagged", "frauds_caught", "precision", "lift_vs_base_rate"])
    print(lift_df.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
 
    avg_fraud_amount = amounts_test[y_test.values == 1].mean()
    avg_fn_c = avg_fraud_amount + COST_ASSUMPTIONS["chargeback_admin_fee"]
 
    print("\n" + "=" * 72)
    print("BREAKEVEN PRECISION - how good the model needs to be to be worth using")
    print("=" * 72)
    for label, fp_c in [
        ("auto-decline (full review+churn cost, $%.0f)" % fp_cost(), fp_cost()),
        ("route to manual review only, no churn risk ($4)", 4.0),
        ("cheap automated secondary signal ($0.50)", 0.50),
    ]:
        be = breakeven_precision(fp_c, tp_cost(), avg_fn_c)
        print(f"  {label:55s} -> needs precision >= {be:.1%}  (best we get at any volume: {lift_df['precision'].max():.1%})")
 
    print("\n" + "=" * 72)
    print("SEGMENT RISK - by merchant_type")
    print("=" * 72)
    print(segment_risk_table(feat, "merchant_type").to_string(float_format=lambda x: f"{x:,.3f}"))
 
    print("\n" + "=" * 72)
    print("SEGMENT RISK - by city")
    print("=" * 72)
    print(segment_risk_table(feat, "city").to_string(float_format=lambda x: f"{x:,.3f}"))
 
    print("\n" + "=" * 72)
    print("SEGMENT RISK - by hour bucket")
    print("=" * 72)
    feat["hour_bucket"] = pd.cut(feat["hour"], bins=[-1, 5, 11, 17, 23], labels=["night(0-5)", "morning(6-11)", "afternoon(12-17)", "evening(18-23)"])
    print(segment_risk_table(feat, "hour_bucket").to_string(float_format=lambda x: f"{x:,.3f}"))
 

COST ASSUMPTIONS
Cost of a missed fraud (FN)      = transaction amount + $25 admin
Cost of a false positive (FP)    = $11.00  ($4 review + 2% churn risk x $350 CLV)
Cost of a caught fraud (TP)      = $8.00 (investigation/action)

COST-MINIMIZING THRESHOLD
threshold         0.648825
total_cost    63668.740000
fn             1153.000000
fp                0.000000
tp                0.000000
Name: 21746, dtype: float64

MODEL vs CURRENT PROCESS vs DOING NOTHING (test set, ~20% of the year)
Doing nothing (no screening at all):      $63,669
Current process (flag amount >= $75):  $104,213  (flags 4.0% of volume, catches 46/1153 frauds)
Model at cost-minimizing threshold:       $63,669  (flags 0.0% of volume, catches 0/1153 frauds)

Model savings vs doing nothing (this window): $0
Model savings vs current process (this window): $40,544
Annualized (x ~5, since test window is ~1/5 of the year's volume):
  vs doing nothing:  $0/yr
  vs current process: $202,722/yr

WHERE THE MODEL ACTUALLY HAS LI

/var/folders/bt/k7t4mr2j6kq79jv3vbj756z80000gn/T/ipykernel_4713/4218185329.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = feat_df.groupby(group_col)
